# HiveMIND: a GPU for the hive's children

While this notebook runs, every child the hive grows is trained here, on this runtime's GPU, in minutes instead of
the three hours a free GitHub runner takes. When it is closed, the hive trains on GitHub as always: nothing waits for it.

**Once:**
1. *Runtime → Change runtime type →* **L4 GPU** (or A100). A T4 is too small for the 8B model.
2. *Secrets* (the key on the left) → add **`HIVE_GITHUB_TOKEN`**, with *Notebook access* on. Its value: a GitHub
   fine-grained token for the repository **Arnav123-s/HiveMIND only**, permission **Contents: Read and write**
   (github.com → Settings → Developer settings → Fine-grained tokens). It lets this notebook post the trained cells to
   the hive's release. It is never printed.

**Each time:** *Runtime → Run all*, and leave the tab open. It uses your plan's compute units only while it is connected:
an L4 costs about 2 units an hour. It stops by itself after 12 hours with nothing to train.

In [ ]:
# 1. The GPU, the hive's code, and what training needs (Colab's own PyTorch, built for its GPU, is kept)
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
!rm -rf /content/HiveMIND && git clone -q --depth 1 https://github.com/Arnav123-s/HiveMIND /content/HiveMIND
%cd /content/HiveMIND
!grep -viE '^(torch|playwright|matplotlib|pytest|pandas)' requirements.txt > /tmp/needs.txt && pip install -q -r /tmp/needs.txt
!python -c "from team import ternary; print('runtime source:', ternary.source())"

In [ ]:
# 2. Serve: say every few minutes that a GPU is here, and train each child the hive orders
import os
from google.colab import userdata
os.environ["HIVE_GITHUB_TOKEN"] = userdata.get("HIVE_GITHUB_TOKEN")
os.environ["GITHUB_REPOSITORY"] = "Arnav123-s/HiveMIND"
!python training/colab.py serve --hours 12